# The original accumap: similarity MDS (Supplementary Note 3; Supp Fig S5, Supp Table S6; reviewer-response Figs R1-R3, Table R1)

The first version of accumap converted receiver-receiver cosine similarity into distance with a curve calibrated on beads with known positions, and solved a similarity-weighted multidimensional scaling problem started from UMAP. The paper calls it **similarity MDS**, keeps it as a baseline (`baselines/` in this folder; not part of the accumap package) and documents in Supplementary Note 3 why it fails on experimental data. This notebook draws, from cached results:

- **Supp Fig S5**: its ceiling on the experimental data (replicate c58_8 and the others);
- **Supp Table S6**: barcode crosstalk injected into the idealized simulation;

and the figures and table that answer the reviewers' questions about it (in the response to reviewers, not the paper):

- **Fig R1**: the method on the idealized simulation;
- **Fig R2**: empirical vs. physically motivated similarity-distance curves, and calibration transfer across replicates;
- **Fig R3**: initialization sensitivity and error bars across simulation replicates;
- **Table R1**: cosine similarity vs. other count-processing and similarity choices.

The underlying runs were development code and are not rerun here; the notebook needs only the cached files in `paper_results/original_method/` and runs in under a minute.

In [ ]:
import os, json, tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy.optimize import curve_fit
from baselines.simulation import per_bead_mse
from fig_style import make_summary_fig, fit_curve_panel, add_panel_letter

RES = os.path.join("paper_results", "original_method")
FIGURES = "../final_figures"
os.makedirs(FIGURES, exist_ok=True)
TITLE_FS, LABEL_FS, TICK_FS, LETTER_FS = 13, 12, 10, 18


def sig_stars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s."


def bracket(ax, x1, x2, y, h, text, fontsize=TICK_FS):
    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], lw=1.1, c="black")
    ax.text((x1 + x2) / 2, y + h, text, ha="center", va="bottom", fontsize=fontsize)

## Fig R1: similarity MDS on the idealized simulation

(A) One simulated array (10,000 receivers and senders, Gaussian kernel σ = 52 µm), receivers at their true positions colored by squared position error, for UMAP and for similarity MDS. (B) The similarity-to-distance curve fit on 100 calibration beads. (C) Neighborhood recovery over 8 simulated arrays (mean ± SD). Panels are rendered separately and stitched, as in the paper.

In [ ]:
z = np.load(os.path.join(RES, "fig_s5_inputs.npz"))
E_umap, E_fit, X_true = z["E_umap"], z["E_fit"], z["X_true"]
d56 = json.load(open(os.path.join(RES, "m5_m6_results.json")))
tmp = tempfile.mkdtemp()


def diff_func(x, A, B, C, D):
    return D - C * np.log(-(A + B - x) / (A - x))


# Panel A
umap_mse, fit_mse = per_bead_mse(E_umap, X_true), per_bead_mse(E_fit, X_true)
fig_a = make_summary_fig(pd.DataFrame(E_umap, columns=["xcoord", "ycoord"]), pd.DataFrame(E_fit, columns=["xcoord", "ycoord"]),
                         umap_mse, fit_mse, name1=f"UMAP, mean MSE: {int(umap_mse.mean())}",
                         name2=f"similarity MDS, mean MSE: {int(fit_mse.mean())}", cbar_label="MSE", log=False, panel_letter="A")
fig_a.savefig(os.path.join(tmp, "a.png"), dpi=200, bbox_inches="tight"); plt.close(fig_a)

# Panel B: curve fit on 100 calibration beads (rows of the similarity and distance matrices)
sv, dv = z["S_rows"].ravel(), z["D_rows"].ravel()
m = sv > 0
popt, _ = curve_fit(diff_func, sv[m], dv[m], p0=(-5.389e-3, 1.104, 296.2, 783.9), maxfev=20000)
xx = np.arange(0, 1, 0.01)
fig_b = fit_curve_panel(pd.DataFrame({"cosine similarity": sv, "distance": dv}), xx, diff_func(xx, *popt), panel_letter="B")
fig_b.savefig(os.path.join(tmp, "b.png"), dpi=200, bbox_inches="tight"); plt.close(fig_b)

# Panel C: Jaccard distance vs number of neighbors, 8 arrays
xs = np.array(d56["jac_xs"])
fig_c, ax = plt.subplots(figsize=(9, 4))
for method, color, label in [("UMAP", "crimson", "UMAP"), ("accumap (UMAP init)", "royalblue", "similarity MDS")]:
    mean, std = np.array(d56["jac_stats"][method]["mean"]), np.array(d56["jac_stats"][method]["std"])
    ax.plot(xs, mean, lw=2, color=color, label=label)
    ax.fill_between(xs, mean - std, mean + std, color=color, alpha=0.18)
ax.set_xlabel("Number of nearest neighbours (X)", fontsize=12); ax.set_ylabel("Mean Jaccard distance", fontsize=12)
ax.set_ylim(-0.01, 0.5)
ax.legend(fontsize=10, frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1))
ax.tick_params(labelsize=10); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
fig_c.tight_layout(); add_panel_letter(fig_c, "C")
fig_c.savefig(os.path.join(tmp, "c.png"), dpi=200, bbox_inches="tight"); plt.close(fig_c)


def h_concat(ims, pad=0):
    h = min(i.height for i in ims); ims = [i.resize((round(i.width * h / i.height), h), Image.LANCZOS) for i in ims]
    out = Image.new("RGB", (sum(i.width for i in ims) + pad * (len(ims) - 1), h), (255, 255, 255)); x = 0
    for i in ims:
        out.paste(i, (x, 0)); x += i.width + pad
    return out


def v_concat(ims, pad=0):
    w = max(i.width for i in ims); ims = [i.resize((w, round(i.height * w / i.width)), Image.LANCZOS) for i in ims]
    out = Image.new("RGB", (w, sum(i.height for i in ims) + pad * (len(ims) - 1)), (255, 255, 255)); y = 0
    for i in ims:
        out.paste(i, (0, y)); y += i.height + pad
    return out


panel = lambda k: Image.open(os.path.join(tmp, f"{k}.png")).convert("RGB")
s5 = v_concat([panel("a"), h_concat([panel("b"), panel("c")], pad=20)], pad=20)
s5.save(os.path.join(FIGURES, "FigR1_final.png"), dpi=(200, 200))
s5

## Supp Fig S5: similarity MDS on the experimental data

(A) Pairwise-distance RMSE on c58_8 for UMAP, the full weighted objective, the objective restricted to each bead's five most similar neighbors, and the solver given the true distances. (B) The neighbor-graph size. (C) The read-depth noise floor: true distance to neighbors identified from half of the reads, against the true nearest neighbors. (D) The same comparison on every replicate.

In [ ]:
d = json.load(open(os.path.join(RES, "m7_m9_data.json")))
C_UMAP, C_FULL, C_KNN, C_ORACLE = "crimson", "gray", "royalblue", "seagreen"
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
axA, axB, axC, axD = axes[0, 0], axes[0, 1], axes[1, 0], axes[1, 1]

# ---------------------------------------------------------------- Panel A --
# reconstruction quality ladder: UMAP / full-weight / kNN(k=5) / oracle
ladder = d["ladder"]
names = list(ladder.keys())
labels = ["UMAP", "similarity MDS\n(full)", "similarity MDS\n(5-NN)",
         "similarity MDS\n(true\ndistances)"]
cols = [C_UMAP, C_FULL, C_KNN, C_ORACLE]
pair_vals = [ladder[n][1] for n in names]
xpos = np.arange(4)
axA.bar(xpos, pair_vals, color=cols, width=0.6)
axA.set_yscale("log")
axA.set_xticks(xpos); axA.set_xticklabels(labels, fontsize=TICK_FS - 1)
axA.set_ylabel("Pairwise-distance RMSE ($\\mu$m, log scale)", fontsize=LABEL_FS)
axA.tick_params(axis="both", labelsize=TICK_FS)
for x, v in zip(xpos, pair_vals):
    axA.annotate(f"{v:.1f}", xy=(x, v), xytext=(0, 4), textcoords="offset points",
                ha="center", fontsize=TICK_FS - 1)
axA.spines["top"].set_visible(False); axA.spines["right"].set_visible(False)

# ---------------------------------------------------------------- Panel B --
# k-sweep: pairRMSE vs k, UMAP reference line, optimum marked
ks = d["ksweep"]
axB.plot(ks["k"], ks["pairRMSE"], "o-", color=C_KNN, lw=2, ms=5, label="similarity MDS (k-NN graph)")
axB.axhline(ks["umap_pairRMSE"], color=C_UMAP, ls="--", lw=1.8, label="UMAP")
k5i = ks["k"].index(5)
axB.scatter([5], [ks["pairRMSE"][k5i]], s=90, facecolors="none", edgecolors="black",
           linewidths=1.5, zorder=5, label="chosen optimum (k=5)")
axB.set_xlabel("k (nearest-neighbour graph size)", fontsize=LABEL_FS)
axB.set_ylabel("Pairwise-distance RMSE ($\\mu$m)", fontsize=LABEL_FS)
axB.tick_params(axis="both", labelsize=TICK_FS)
axB.legend(fontsize=TICK_FS - 1, frameon=False)
axB.spines["top"].set_visible(False); axB.spines["right"].set_visible(False)

# ---------------------------------------------------------------- Panel C --
# split-half noise floor
sh = d["splithalf"]
xk = np.arange(len(sh["k"]))
w = 0.35
axC.bar(xk - w / 2, sh["oracle_true_nn_dist"], width=w, color=C_ORACLE,
       label="oracle (true nearest\nneighbours, imaging)")
axC.bar(xk + w / 2, sh["half_depth_picked_dist"], width=w, color="darkorange",
       label="picked from half-depth\nreads (noise only)")
axC.set_xticks(xk); axC.set_xticklabels([f"k={k}" for k in sh["k"]], fontsize=TICK_FS)
axC.set_ylabel("Mean true distance to\nidentified neighbours ($\\mu$m)", fontsize=LABEL_FS)
axC.tick_params(axis="both", labelsize=TICK_FS)
axC.legend(fontsize=TICK_FS - 1, frameon=False)
axC.spines["top"].set_visible(False); axC.spines["right"].set_visible(False)

# ---------------------------------------------------------------- Panel D --
# cross-replicate consistency
cr = d["cross_replicate"]
reps = list(cr.keys())
xr = np.arange(len(reps))
wr = 0.3
axD.bar(xr - wr / 2, [cr[r]["umap"][1] for r in reps], width=wr, color=C_UMAP, label="UMAP")
axD.bar(xr + wr / 2, [cr[r]["accumap_k5"][1] for r in reps], width=wr, color=C_KNN,
       label="similarity MDS (5-NN)")
axD.set_xticks(xr); axD.set_xticklabels(reps, fontsize=TICK_FS)
axD.set_ylabel("Pairwise-distance RMSE ($\\mu$m)", fontsize=LABEL_FS)
axD.tick_params(axis="both", labelsize=TICK_FS)
axD.legend(fontsize=TICK_FS - 1, frameon=False)
axD.spines["top"].set_visible(False); axD.spines["right"].set_visible(False)

for ax, letter in [(axA, "A"), (axB, "B"), (axC, "C"), (axD, "D")]:
    ax.text(-0.14, 1.08, letter, transform=ax.transAxes, fontsize=LETTER_FS,
           fontweight="bold", va="top", ha="left")

fig.tight_layout()
fig.savefig(os.path.join(FIGURES, "FigS5_final.png"), dpi=300)
plt.show()

## Fig R2: the similarity-distance curve

(A) Held-out accuracy of the empirical curve against a physically motivated inverse with no fitted parameters, and (B) the resulting reconstructions, over 8 simulated arrays (mean ± SEM). (C) A curve calibrated on c58_8 and transferred to the other replicates without their imaging, against a curve fit to each replicate's own imaging.

In [ ]:
C_UMAP, C_EMP, C_PHYS, C_FRESH, C_TRANS = "crimson", "royalblue", "gray", "royalblue", "mediumpurple"
d3 = json.load(open(os.path.join(RES, "m3_deep_results.json")))
dx = json.load(open(os.path.join(RES, "cross_replicate_transfer.json")))
n = d3["n_seeds"]




fig = plt.figure(figsize=(14, 4.6))
axA = fig.add_subplot(1, 3, 1)
axB = fig.add_subplot(1, 3, 2)
axC = fig.add_subplot(1, 3, 3)

# ---------------------------------------------------------------- Panel A --
# held-out curve-fit RMSE: empirical (4-param, fitted) vs physical (0-param)
rmse_emp = np.array(d3["rmse_emp_holdout"])
rmse_phys = np.array(d3["rmse_phys_holdout"])
means = [rmse_emp.mean(), rmse_phys.mean()]
sems = [rmse_emp.std(ddof=1) / np.sqrt(n), rmse_phys.std(ddof=1) / np.sqrt(n)]
xpos = np.arange(2)
axA.bar(xpos, means, yerr=sems, capsize=5, color=[C_EMP, C_PHYS], width=0.6,
       error_kw=dict(lw=1.3))
axA.set_xticks(xpos)
axA.set_xticklabels(["empirical curve\n(4 fitted params)", "physical inverse\n(0 params)"],
                    fontsize=TICK_FS)
axA.set_ylabel("Held-out curve RMSE ($\\mu$m)", fontsize=LABEL_FS)
axA.tick_params(axis="both", labelsize=TICK_FS)
p = d3["ttest_rmse_emp_vs_phys"]["p"]
ytop = max(means[0] + sems[0], means[1] + sems[1])
bracket(axA, 0, 1, means[1] + sems[1] * 1.5, ytop * 0.05, f"p = {p:.1e} ({sig_stars(p)})")
axA.set_ylim(0, means[1] * 1.35)
axA.set_title(f"Curve-fit accuracy (n={n})", fontsize=TITLE_FS)
axA.spines["top"].set_visible(False); axA.spines["right"].set_visible(False)

# ---------------------------------------------------------------- Panel B --
# downstream reconstruction MSE: UMAP / empirical / physical
methB = ["UMAP", "accumap (empirical curve)", "accumap (physical inverse)"]
labB = ["UMAP", "similarity MDS\n(empirical curve)", "similarity MDS\n(physical inverse)"]
colsB = [C_UMAP, C_EMP, C_PHYS]
mse = {k: np.array(v) for k, v in d3["mse_by_method"].items()}
meansB = [mse[m].mean() for m in methB]
semsB = [mse[m].std(ddof=1) / np.sqrt(n) for m in methB]
xposB = np.arange(3)
axB.bar(xposB, meansB, yerr=semsB, capsize=5, color=colsB, width=0.6, error_kw=dict(lw=1.3))
axB.set_xticks(xposB)
axB.set_xticklabels(labB, fontsize=TICK_FS)
axB.set_ylabel("Mean per-bead MSE ($\\mu$m$^2$)", fontsize=LABEL_FS)
axB.tick_params(axis="both", labelsize=TICK_FS)
pB = d3["ttest_downstream_phys_vs_emp"]["p"]
bracket(axB, 1, 2, meansB[2] + semsB[2] * 1.3, max(meansB) * 0.05, f"p = {pB:.1e} ({sig_stars(pB)})")
axB.set_ylim(0, meansB[2] * 1.35)
axB.set_title(f"Downstream reconstruction (n={n})", fontsize=TITLE_FS)
axB.spines["top"].set_visible(False); axB.spines["right"].set_visible(False)

# ---------------------------------------------------------------- Panel C --
# cross-replicate transfer: UMAP / fresh curve / transferred curve, per replicate
reps = list(dx["results"].keys())
width = 0.25
xposC = np.arange(len(reps))
umap_v = [dx["results"][r]["umap"][1] for r in reps]   # pairRMSE
fresh_v = [dx["results"][r]["fresh"][1] for r in reps]
trans_v = [dx["results"][r]["transfer"][1] for r in reps]
axC.bar(xposC - width, umap_v, width=width, color=C_UMAP, label="UMAP")
axC.bar(xposC, fresh_v, width=width, color=C_FRESH,
       label="fresh curve\n(own imaging)")
axC.bar(xposC + width, trans_v, width=width, color=C_TRANS,
       label="transferred curve\n(from c58_8, no imaging)")
axC.set_xticks(xposC)
axC.set_xticklabels(reps, fontsize=TICK_FS)
axC.set_ylabel("Pairwise-distance RMSE ($\\mu$m)", fontsize=LABEL_FS)
axC.tick_params(axis="both", labelsize=TICK_FS)
axC.legend(fontsize=TICK_FS - 1, frameon=False, loc="upper left")
axC.set_title("Calibration transfer across replicates", fontsize=TITLE_FS)
axC.spines["top"].set_visible(False); axC.spines["right"].set_visible(False)
axC.set_ylim(0, max(umap_v + fresh_v + trans_v) * 1.35)

for ax, letter in [(axA, "A"), (axB, "B"), (axC, "C")]:
    ax.text(-0.12, 1.08, letter, transform=ax.transAxes, fontsize=LETTER_FS,
           fontweight="bold", va="top", ha="left")

fig.tight_layout()
fig.savefig(os.path.join(FIGURES, "FigR2_final.png"), dpi=300)
plt.show()

## Fig R3: error bars and initialization sensitivity

Over 8 simulated arrays: (A) mean squared position error of UMAP and similarity MDS (mean ± SEM, paired t-test), (B) neighborhood recovery (mean ± SD), and (C) similarity MDS started from UMAP, classical multidimensional scaling or random positions.

In [ ]:
C_UMAP, C_ACCU, C_RAND, C_CMDS = "crimson", "royalblue", "darkorange", "seagreen"
d = d56
n = d["n_seeds"]
mse = {k: np.array(v) for k, v in d["mse_by_method"].items()}
xs = np.array(d["jac_xs"])
jac = d["jac_stats"]




fig = plt.figure(figsize=(14, 4.6))
axA = fig.add_subplot(1, 3, 1)
axB = fig.add_subplot(1, 3, 2)
axC = fig.add_subplot(1, 3, 3)

# ---------------------------------------------------------------- Panel A --
# M6: UMAP vs accumap(UMAP init), mean +/- SEM, paired-test annotation
methA = ["UMAP", "accumap (UMAP init)"]
means = [mse[m].mean() for m in methA]
sems = [mse[m].std(ddof=1) / np.sqrt(n) for m in methA]
colsA = [C_UMAP, C_ACCU]
xpos = np.arange(2)
axA.bar(xpos, means, yerr=sems, capsize=5, color=colsA, width=0.6,
       error_kw=dict(lw=1.3))
axA.set_xticks(xpos)
axA.set_xticklabels(["UMAP", "similarity MDS"], fontsize=LABEL_FS)
axA.set_ylabel("Mean per-bead MSE ($\\mu$m$^2$)", fontsize=LABEL_FS)
axA.tick_params(axis="both", labelsize=TICK_FS)
p = d["ttest_accumap_vs_umap"]["p"]
ytop = max(means[0] + sems[0], means[1] + sems[1])
bracket(axA, 0, 1, ytop * 1.08, ytop * 0.05,
       f"paired t-test\np = {p:.1e} ({sig_stars(p)})")
axA.set_ylim(0, ytop * 1.35)
axA.set_title(f"Simulation replicates (n={n})", fontsize=TITLE_FS)
axA.spines["top"].set_visible(False); axA.spines["right"].set_visible(False)

# ---------------------------------------------------------------- Panel B --
# M6: Jaccard vs X, mean +/- std shading, UMAP + accumap(UMAP init) only
for m, c in [("UMAP", C_UMAP), ("accumap (UMAP init)", C_ACCU)]:
    mean = np.array(jac[m]["mean"]); std = np.array(jac[m]["std"])
    axB.plot(xs, mean, lw=2, color=c, label="UMAP" if m == "UMAP" else "similarity MDS")
    axB.fill_between(xs, mean - std, mean + std, color=c, alpha=0.18)
axB.set_xlabel("Number of nearest neighbours (X)", fontsize=LABEL_FS)
axB.set_ylabel("Mean Jaccard distance", fontsize=LABEL_FS)
axB.tick_params(axis="both", labelsize=TICK_FS)
axB.legend(fontsize=TICK_FS, frameon=False)
axB.set_title(f"Mean $\\pm$ s.d., n={n} replicates", fontsize=TITLE_FS)
axB.spines["top"].set_visible(False); axB.spines["right"].set_visible(False)

# ---------------------------------------------------------------- Panel C --
# M5: initialization sensitivity, log scale, all 4 conditions
methC = ["UMAP", "accumap (UMAP init)", "accumap (cMDS init)", "accumap (random init)"]
labelsC = ["UMAP", "similarity\nMDS\n(UMAP init)", "similarity\nMDS\n(cMDS init)", "similarity\nMDS\n(random init)"]
colsC = [C_UMAP, C_ACCU, C_CMDS, C_RAND]
meansC = np.array([mse[m].mean() for m in methC])
semsC = np.array([mse[m].std(ddof=1) / np.sqrt(n) for m in methC])
xposC = np.arange(4)
axC.bar(xposC, meansC, yerr=semsC, capsize=5, color=colsC, width=0.6,
       error_kw=dict(lw=1.3))
axC.set_yscale("log")
axC.set_xticks(xposC)
axC.set_xticklabels(labelsC, fontsize=TICK_FS - 1)
axC.set_ylabel("Mean per-bead MSE ($\\mu$m$^2$, log scale)", fontsize=LABEL_FS)
axC.tick_params(axis="both", labelsize=TICK_FS)
p_rand = d["ttest_randinit_vs_umapinit"]["p"]
p_cmds = d["ttest_cmdsinit_vs_umapinit"]["p"]
axC.set_ylim(70, meansC.max() * 25)
for xi, mi, p in [(2, meansC[2], p_cmds), (3, meansC[3], p_rand)]:
    axC.annotate(f"p={p:.1e}\nvs UMAP-init", xy=(xi, mi), xytext=(0, 20),
                textcoords="offset points", ha="center", va="bottom",
                fontsize=TICK_FS - 1)
axC.set_title("Initialization sensitivity", fontsize=TITLE_FS)
axC.spines["top"].set_visible(False); axC.spines["right"].set_visible(False)

for ax, letter in [(axA, "A"), (axB, "B"), (axC, "C")]:
    ax.text(-0.12, 1.08, letter, transform=ax.transAxes, fontsize=LETTER_FS,
           fontweight="bold", va="top", ha="left")

fig.tight_layout()
fig.savefig(os.path.join(FIGURES, "FigR3_final.png"), dpi=300)
plt.show()

## Supp Table S6: injected barcode crosstalk

A fraction μ of each receiver's reads is reassigned to random receivers (same sender) in the idealized simulation. ρ is the Spearman correlation between cosine similarity and true distance; kNN-Jaccard compares each bead's 10 most similar beads with its 10 true nearest neighbors. Mean squared errors are per bead after alignment; ratios are relative to UMAP. The full objective was run only up to μ = 0.05.

In [ ]:
ct = json.load(open(os.path.join(RES, "sim_crosstalk_sweep.json")))
rows = []
for mu, r in ct.items():
    full = r["full_mse_mean"] if r["full_mse_mean"] == r["full_mse_mean"] else None   # NaN: not run
    rows.append({"mu": float(mu), "rho": r["rho"], "kNN-Jaccard (X=10)": r["kNN_jac10"], "UMAP MSE": r["umap_mse_mean"],
                 "full-weight MSE": full, "full/UMAP": r["full_vs_umap_mse_ratio"],
                 "hard kNN MSE (k=5)": r["knn5_mse_mean"], "kNN/UMAP": r["knn5_vs_umap_mse_ratio"]})
pd.DataFrame(rows).set_index("mu").round({"rho": 2, "kNN-Jaccard (X=10)": 3, "UMAP MSE": 1, "full-weight MSE": 1,
                                          "full/UMAP": 2, "hard kNN MSE (k=5)": 1, "kNN/UMAP": 2})

## Table R1: cosine similarity vs. alternatives

Rows 1-6: the idealized simulation reduced to 3,000 receivers and senders. Last rows: log(1 + C) vs. raw counts before cosine similarity, at full size over 8 arrays (mean ± SEM).

In [ ]:
sm = json.load(open(os.path.join(RES, "sim_similarity_metric_test.json")))
lr = json.load(open(os.path.join(RES, "log1p_vs_raw_8rep.json")))
rows = [{"test": "UMAP embedding", "condition": k, "mean MSE (um^2)": round(v["mse_mean"], 1), "J10": round(v["jac10"], 3)}
        for k, v in sm["umap_variants"].items()]
rows += [{"test": "similarity MDS reconstruction", "condition": f"{k} similarity", "mean MSE (um^2)": round(sm[f"accumap_{k}"]["mse_mean"], 1),
          "J10": round(sm[f"accumap_{k}"]["jac10"], 3)} for k in ("cosine", "L1", "L2")]
for method in ("umap", "accumap"):
    for counts in ("raw", "log1p"):
        v = np.array([r[f"{method}_mse"] for r in lr["results"][counts]])
        rows.append({"test": "log1p vs raw counts (n = 8)", "condition": f"{'UMAP' if method == 'umap' else 'similarity MDS'}, {counts} counts",
                     "mean MSE (um^2)": f"{v.mean():.1f} ± {v.std(ddof=1) / np.sqrt(len(v)):.1f}", "J10": None})
pd.DataFrame(rows).set_index(["test", "condition"])